# **Medicinal Chemistry Hygiene Filtering of Generated Molecules**

This notebook performs a medicinal chemistry hygiene filter on generated molecules after pIC50 prediction. The goal is to remove chemically invalid structures, flag problematic substructures, and identify compounds with physicochemical properties compatible with drug-like space.

The workflow applies a series of lightweight medicinal chemistry triage checks, including SMILES validity, physicochemical property calculations, structural alert screening, and rule-based filters inspired by common drug-likeness guidelines. These steps help prioritize compounds that are more likely to be synthetically tractable and biologically viable while reducing the risk of selecting molecules prone to assay interference, toxicity, or poor ADME behavior.


## **What each step does & why it matters**



1. Validity (parse SMILES) - redundant, but no harm

    - What: Remove strings that RDKit cannot parse.

    - Why: Invalid structures break all later steps.

2. Physicochemical descriptors (MW, cLogP, RB (Rotatable bonds), TPSA, HBD/HBA)

    - What: Compute simple properties linked to absorption/permeability/solubility.

    - Why: Compounds far outside Ro5/Veber-like ranges (Rotatable Bonds ≤ 10, TPSA ≤ 140 Å²) are frequent ADME failures.

3. Structural alerts (PAINS/BRENK/NIH)

    - What: Flag substructures associated with assay interference/reactivity/tox.

    - Why: Reduces wasted effort on likely false positives or problematic chemotypes.

    Note: Treat as flags; some true actives can trigger alerts.

4. Soft gates (general or CNS-biased)

    - What: Turn continuous descriptors into simple pass/fail band checks.

    - Why: Speeds triage and standardizes decisions; tunable to project needs.

    CNS option: Stricter TPSA/MW/logP to favor BBB penetration.

5. Explainability (“Why” column)

    - What: Human-readable reasons for failure (e.g., “TPSA high; Structural alert”).

    - Why: Makes the screen auditable, reversible, and easy to communicate.

6. Outputs

    - admet_hygiene_annotated.csv → master table with descriptors, flags, and reasons.

    - admet_hygiene_pass.smi → quick list of “green” molecules  (ADMET/tox models; ).

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
!pip install rdkit

In [ ]:
# ============================
# Med-chem hygiene
# ============================
# WHAT: Quick “sanity” filtering of a generated library using RDKit.
# WHY: Remove obvious non-starters and annotate risks before heavier ADMET/tox/potency work.

import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors as rdMD
from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")  #  silence RDKit warnings to keep outputs readable

In [ ]:
from pathlib import Path
from datetime import datetime
import pandas as pd

targets = ["5-HT6","ache","bace1","buche","esr1","3beta","mao-b"]
target = targets[3]

# ============================
# Paths (for pIC50-filtered set)
# ============================
BASE = Path("/content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/GenAI/Filtering")

# Input folder from Step 1 (potency filtering)
IN_DIR = BASE / "1-pIC5_5_filtered_mols"

# Output folder for MedChem hygiene step (Step 2)
OUT_BASE = BASE / "2-medchem_hygiene"

# Make timestamped run directory per target
run_id = datetime.now().strftime("%Y%m%d-%H%M%S")
OUTDIR = OUT_BASE / f"{target}_hygiene-{run_id}"
OUTDIR.mkdir(parents=True, exist_ok=True)

# Input file (mols filterd by pIC50>5)
in_csv = IN_DIR / f"{target}_genai_1k_filtered_pred_pIC50_ge_5.0.csv"

# Outputs
ann_csv  = OUTDIR / "medchem_hygiene_annotated.csv"
pass_smi = OUTDIR / "medchem_hygiene_pass.smi"
pass_csv = OUTDIR / "medchem_hygiene_pass.csv"

print("Input :", in_csv)
print("OUTDIR:", OUTDIR)
print("Annotated:", ann_csv)
print("Pass SMI :", pass_smi)
print("Pass CSV :", pass_csv)




Input : /content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/GenAI/Filtering/1-pIC5_5_filtered_mols/mao-b_genai_1k_filtered_pred_pIC50_ge_5.0.csv
OUTDIR: /content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/GenAI/Filtering/2-medchem_hygiene/mao-b_hygiene-20251225-094837
Annotated: /content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/GenAI/Filtering/2-medchem_hygiene/mao-b_hygiene-20251225-094837/medchem_hygiene_annotated.csv
Pass SMI : /content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/GenAI/Filtering/2-medchem_hygiene/mao-b_hygiene-20251225-094837/medchem_hygiene_pass.smi
Pass CSV : /content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/GenAI/Filtering/2-medchem_hygiene/mao-b_hygiene-20251225-094837/medchem_hygiene_pass.csv


'# ---- 0) Load your potency-filtered set ----\ndf = pd.read_csv(in_csv)\n\n# Robustly locate the SMILES column (handles SMILES vs canonical_smiles)\nsmiles_col = "smiles" if "smiles" in df.columns else ("canonical_smiles" if "canonical_smiles" in df.columns else None)\nif smiles_col is None:\n    raise KeyError(f"No SMILES column found. Columns available: {df.columns.tolist()}")\n\ndf = df.dropna(subset=[smiles_col]).copy()\ndf.rename(columns={smiles_col: "smiles"}, inplace=True)\n\nprint("Loaded rows:", len(df))\nprint(df[["smiles"]].head()) '

In [ ]:

# ---- 0) Load Step 1 potency-filtered set ----
df = pd.read_csv(in_csv)

# --- Normalize SMILES column to "SMILES"  ---
# accept common variants
smiles_candidates = ["SMILES", "smiles", "canonical_smiles", "Canonical_SMILES"]
smiles_col = next((c for c in smiles_candidates if c in df.columns), None)
if smiles_col is None:
    raise KeyError(f"No SMILES column found. Columns available: {df.columns.tolist()}")

df = df.dropna(subset=[smiles_col]).copy()
df.rename(columns={smiles_col: "SMILES"}, inplace=True)

# --- Normalize prediction column to "pred_pIC50" ---
# If file already has pred_pIC50, this keeps it.
pred_candidates = [
    "pred_pIC50", "pIC50_pred", "predicted_pIC50", "y_pred", "pred", "pIC50"
]
pred_col = next((c for c in pred_candidates if c in df.columns), None)

if pred_col is None:
    # try : any column containing 'pic50' and 'pred'
    fuzzy = [c for c in df.columns if ("pic50" in c.lower() and "pred" in c.lower())]
    if len(fuzzy) == 1:
        pred_col = fuzzy[0]
    else:
        print(" No prediction column found to rename → pred_pIC50. Continuing without it.")
        pred_col = None

if pred_col is not None and pred_col != "pred_pIC50":
    df.rename(columns={pred_col: "pred_pIC50"}, inplace=True)

# Optional: keep only core columns + whatever else needed
# (don’t do this if  need extra metadata later)
# keep_cols = ["SMILES"] + (["pred_pIC50"] if "pred_pIC50" in df.columns else [])
# df = df[keep_cols].copy()

print("Loaded rows:", len(df))
print("Columns:", df.columns.tolist())
print(df[["SMILES"] + (["pred_pIC50"] if "pred_pIC50" in df.columns else [])].head())


Loaded rows: 895
Columns: ['SMILES', 'pred_pIC50']
                                 SMILES  pred_pIC50
0  Cc1c(C)c2ccc(OCc3ccc(Br)cc3)cc2oc1=O    9.011366
1   Cc1ccc(NC(=O)c2ccc3c(cnn3C)c2)cc1Cl    8.962576
2   Cc1ccc(COc2ccc3c(CO)cc(=O)oc3c2)cc1    8.861387
3     O=C(Nc1cccc(Cl)c1)c1coc2sccc2c1=O    8.650835
4            COc1ccc2c(C)c(C)c(=O)oc2c1    8.631010


In [ ]:
# ---- 1) Build structural-alert catalogs (PAINS, BRENK, NIH) ----
# WHAT: Collections of substructures known to cause assay interference, reactivity, or tox liabilities.
# WHY: Early warning; not absolute “kill” switches, but strong caution flags for medicinal chemistry.
# Robust import for recent RDKit versions
from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams

# Build a combined catalog (PAINS A/B, BRENK, NIH)
params = FilterCatalogParams()
params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS_A)
params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS_B)
params.AddCatalog(FilterCatalogParams.FilterCatalogs.BRENK)
params.AddCatalog(FilterCatalogParams.FilterCatalogs.NIH)

catalog = FilterCatalog(params)

# Quick test
from rdkit import Chem
m = Chem.MolFromSmiles("Oc1ccccc1")  # phenol
print("Has alert?", bool(catalog.HasMatch(m)))



Has alert? False


In [ ]:
rows = []
for s in df["SMILES"].astype(str):
    # ---- 2) Parse & basic validity ----
    # WHAT: Convert SMILES → RDKit Mol; invalid strings are dropped from consideration.
    # WHY: Invalid SMILES cannot be synthesized/assessed; they are noise for all downstream steps.
    m = Chem.MolFromSmiles(s)
    if m is None:
        rows.append({"smiles": s, "valid": False, "why_invalid": "SMILES parse failed"})
        continue

    # ---- 3) Compute fast physicochemical descriptors ----
    # WHAT: MW, cLogP, rotatable bonds, polar surface area, HBD/HBA.
    # WHY: These correlate with permeability, solubility, and oral exposure (Ro5/Veber heuristics).
    mw   = Descriptors.MolWt(m)
    clogp= Descriptors.MolLogP(m)                # NOTE: later may prefer logD7.4 for ionizables
    rb   = rdMD.CalcNumRotatableBonds(m)
    tpsa = rdMD.CalcTPSA(m)
    hbd  = rdMD.CalcNumHBD(m)
    hba  = rdMD.CalcNumHBA(m)

    # ---- 4) Structural alerts ----
    # WHAT: Does the molecule match any PAINS/BRENK/NIH patterns?
    # WHY: Alerts often mark false positives, reactive motifs, or frequent hitters; flag for review.
    has_alert = catalog.HasMatch(m)

    rows.append({
        "SMILES": s, "valid": True,
        "MW": mw, "cLogP": clogp, "RB": rb, "TPSA": tpsa, "HBD": hbd, "HBA": hba,
        "Alert": bool(has_alert)
    })

out = pd.DataFrame(rows)



In [ ]:
# ---- 5) Define soft gates  ----
# WHAT: literature-inspired ranges (Ro5/Veber) used as *soft* acceptability bands.
# WHY: Compounds far outside these zones often suffer from poor PK/absorption; flag them early.
SOFT = dict(
    MW_max   = 550,          # Lipinski trend (≤500) + small buffer
    cLogP_lo = 0.5,          # avoid very hydrophilic compounds (solubility vs permeability balance)
    cLogP_hi = 5.5,          # avoid very greasy compounds (insolubility/clearance issues)
    RB_max   = 10,           # Veber: flexibility limit (oral bioavailability)
    TPSA_max = 140           # Veber: permeability proxy (membrane crossing)
)

# (Optional) CNS-biased caps, if target is central (e.g., 5-HT6, BACE1)
CNS = dict(
    TPSA_max = 90,           # commonly used for BBB penetration
    MW_max   = 450,
    cLogP_lo = 2.0,          # CNS often benefits from moderate lipophilicity
    cLogP_hi = 3.8
)

USE_CNS = True  # Enable CNS-specific filtering (stricter size and polarity limits to favor BBB penetration)


In [ ]:

# ---- 6) Apply gates to produce pass/fail flags ----
# WHAT: Create boolean “passes” for general med-chem hygiene (and optional CNS).
# WHY: Turn continuous descriptors into simple decisions that can drive triage.
soft_pass = (
    (out["valid"]) &
    (out["MW"]   <= (CNS["MW_max"]   if USE_CNS else SOFT["MW_max"])) &
    (out["cLogP"].between(
        (CNS["cLogP_lo"] if USE_CNS else SOFT["cLogP_lo"]),
        (CNS["cLogP_hi"] if USE_CNS else SOFT["cLogP_hi"])
    )) &
    (out["RB"]   <= (CNS["MW_max"]//55 if USE_CNS else SOFT["RB_max"])) &  #  keep SOFT for most cases
    (out["TPSA"] <= (CNS["TPSA_max"] if USE_CNS else SOFT["TPSA_max"]))
)

# Final boolean incorporating structural alerts:
# WHAT: mark molecules that pass property bands *and* have no alerts as immediate “Pass”.
# WHY: Alerts are strong risk indicators; default to caution. You can loosen this later if needed.
out["MedChem_Pass"] = soft_pass & (~out["Alert"])

In [ ]:

# ---- 7) Explainability: why did a molecule fail? ----
# WHAT: Human-readable reasons
# WHY: Transparent decisions enable sensible overrides and rapid iteration with med-chem input.
def reasons(row):
    if not row["valid"]:
        return "Invalid SMILES"
    msgs = []
    if row["MW"]   > (CNS["MW_max"]   if USE_CNS else SOFT["MW_max"]):   msgs.append("MW high")
    if not ((CNS["cLogP_lo"] if USE_CNS else SOFT["cLogP_lo"]) <= row["cLogP"] <= (CNS["cLogP_hi"] if USE_CNS else SOFT["cLogP_hi"])):
        msgs.append("cLogP out of range")
    if row["RB"]   > (SOFT["RB_max"] if not USE_CNS else 10):            msgs.append("RB high")
    if row["TPSA"] > (CNS["TPSA_max"] if USE_CNS else SOFT["TPSA_max"]): msgs.append("TPSA high")
    if row["Alert"]:                                                     msgs.append("Structural alert")
    return "; ".join(msgs) or "OK"

out["Why"] = out.apply(reasons, axis=1)




In [ ]:
# ---- 8) Save annotated table and a filtered SMI (optional) ----
# WHAT: Keep a rich CSV with all descriptors + flags; also write a quick SMI of immediate passes.
# WHY: The CSV is the master record; the SMI is a handy input for downstream ADMET/tox tools.
out.to_csv("admet_hygiene_annotated.csv", index=False)

passed = out[out["MedChem_Pass"]].copy()
passed[["SMILES"]].to_csv("admet_hygiene_pass.smi", index=False, header=False)

print(f"Annotated → admet_hygiene_annotated.csv")
print(f"Pass (no alerts + within soft bands): {len(passed)}/{len(out)} → admet_hygiene_pass.smi")

Annotated → admet_hygiene_annotated.csv
Pass (no alerts + within soft bands): 327/895 → admet_hygiene_pass.smi


In [ ]:
import os
from pathlib import Path
import pandas as pd

# 0) Verify DataFrame and column exist
print("out shape/cols:", out.shape, list(out.columns)[:15])
assert "MedChem_Pass" in out.columns, "MedChem_Pass column not present in `out`"
assert "SMILES" in out.columns, "SMILES column not present in `out`"

# ================================
# Bring in predicted pIC50 (if missing)
# ================================
if "pred_pIC50" not in out.columns:
    STEP1_PRED_CSV = Path(
        f"/content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/GenAI/Filtering/1-pIC5_5_filtered_mols/{target}_genai_1k_filtered_pred_pIC50_ge_5.0.csv"
    )
    preds = pd.read_csv(STEP1_PRED_CSV)

    # normalize SMILES col in preds if needed
    if "SMILES" not in preds.columns:
        if "canonical_smiles" in preds.columns:
            preds = preds.rename(columns={"canonical_smiles": "SMILES"})
        elif "smiles" in preds.columns:
            preds = preds.rename(columns={"smiles": "SMILES"})
        else:
            raise KeyError(f"No SMILES column found in {STEP1_PRED_CSV}. Columns: {preds.columns.tolist()}")

    # normalize pred col in preds
    if "pred_pIC50" not in preds.columns:
        cand = [c for c in preds.columns if "pic50" in c.lower() and ("pred" in c.lower() or c.lower().startswith("y_"))]
        if len(cand) == 1:
            preds = preds.rename(columns={cand[0]: "pred_pIC50"})
        else:
            raise KeyError(f"Couldn't find pred pIC50 column in {STEP1_PRED_CSV}. Columns={preds.columns.tolist()}")

    preds = preds[["SMILES", "pred_pIC50"]].dropna(subset=["SMILES"]).drop_duplicates("SMILES")

    # merge into out (keep all rows in out)
    out = out.merge(preds, on="SMILES", how="left")

    print("Missing pred_pIC50 after merge:", out["pred_pIC50"].isna().sum())
else:
    print(" pred_pIC50 already present in `out` — no merge needed.")

# 1) Rebuild absolute paths
OUTDIR = Path(OUTDIR).resolve()
OUTDIR.mkdir(parents=True, exist_ok=True)
ann_csv   = (OUTDIR / "medchem_hygiene_annotated.csv").resolve()
pass_smi  = (OUTDIR / "medchem_hygiene_pass.smi").resolve()
pass_csv  = (OUTDIR / "medchem_hygiene_pass.csv").resolve()
fails_csv = (OUTDIR / "medchem_hygiene_fails_with_reasons.csv").resolve()

print("CWD      :", os.getcwd())
print("OUTDIR   :", OUTDIR)
print("ann_csv  :", ann_csv)
print("pass_smi :", pass_smi)
print("pass_csv :", pass_csv)

# 2) Save using absolute paths
out.to_csv(str(ann_csv), index=False)

passed = out[out["MedChem_Pass"]].copy()
passed.to_csv(str(pass_csv), index=False)                       # pass set WITH pred_pIC50
passed[["SMILES"]].to_csv(str(pass_smi), index=False, header=False)

out[~out["MedChem_Pass"]].to_csv(str(fails_csv), index=False)

# 3) Confirm on disk
print("Exists ann_csv? ", ann_csv.exists(), ann_csv)
print("Exists pass_csv? ", pass_csv.exists(), pass_csv)
print("Exists pass_smi?", pass_smi.exists(), pass_smi)
print("Exists fails?   ", fails_csv.exists(), fails_csv)

# 4) check what was saved
print("\nSaved columns in ann_csv head:")
print(pd.read_csv(ann_csv, nrows=2).columns.tolist())


out shape/cols: (895, 11) ['SMILES', 'valid', 'MW', 'cLogP', 'RB', 'TPSA', 'HBD', 'HBA', 'Alert', 'MedChem_Pass', 'Why']
Missing pred_pIC50 after merge: 0
CWD      : /content
OUTDIR   : /content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/GenAI/Filtering/2-medchem_hygiene/mao-b_hygiene-20251225-094837
ann_csv  : /content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/GenAI/Filtering/2-medchem_hygiene/mao-b_hygiene-20251225-094837/medchem_hygiene_annotated.csv
pass_smi : /content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/GenAI/Filtering/2-medchem_hygiene/mao-b_hygiene-20251225-094837/medchem_hygiene_pass.smi
pass_csv : /content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/GenAI/Filtering/2-medchem_hygiene/mao-b_hygiene-20251225-094837/medchem_hygiene_pass.csv
Exists ann_csv?  True /content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/GenAI/Filtering/2-medchem_hygiene/mao-b_hygiene-20251225-094837/medchem_hygiene_annotated.csv
Exists pa

In [ ]:
# Assuming already have: OUTDIR, out (annotated DataFrame), and `passed` (subset)
pass_csv = OUTDIR / "medchem_hygiene_pass.csv"  # NEW: CSV version of the pass set

# Save the full annotations for the passing molecules
cols_keep = ["SMILES", "MW", "cLogP", "RB", "TPSA", "HBD", "HBA", "Alert", "MedChem_Pass", "Why"]
(passed[cols_keep] if all(c in passed.columns for c in cols_keep) else passed).to_csv(pass_csv, index=False)

# already save the SMI:
# passed[["SMILES"]].to_csv(pass_smi, index=False, header=False)

print(f"Pass CSV  → {pass_csv}")
print(f"Pass SMI  → {pass_smi}")


Pass CSV  → /content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/GenAI/Filtering/2-medchem_hygiene/mao-b_hygiene-20251225-094837/medchem_hygiene_pass.csv
Pass SMI  → /content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/GenAI/Filtering/2-medchem_hygiene/mao-b_hygiene-20251225-094837/medchem_hygiene_pass.smi


In [ ]:
from pathlib import Path
import pandas as pd
from datetime import datetime

# Reuse OUTDIR, in_csv, ann_csv, pass_smi from your notebook
# If needed, set them explicitly:
# OUTDIR   = Path(".../reinvent/<target>/samples/analyses/hygiene-<timestamp>")
# in_csv   = Path(".../reinvent/<target>/samples/clean_smiles_1k.csv")
# ann_csv  = OUTDIR / "admet_hygiene_annotated.csv"
# pass_smi = OUTDIR / "admet_hygiene_pass.smi"

ann = pd.read_csv(ann_csv)
n_input = len(pd.read_csv(in_csv))
n_valid = int(ann["valid"].sum())
n_pass  = int(ann["MedChem_Pass"].sum())
n_alert = int(ann["Alert"].sum())

summary = pd.DataFrame([{
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "target": str(OUTDIR).split("/")[-3],            # crude extraction of <target>
    "input_file": str(in_csv),
    "annotated_file": str(ann_csv),
    "pass_smi_file": str(pass_smi),
    "n_input": n_input,
    "n_valid": n_valid,
    "n_pass": n_pass,
    "n_alert_flagged": n_alert,
    "pass_rate_vs_input": round(n_pass / max(1, n_input), 3),
    "pass_rate_vs_valid": round(n_pass / max(1, n_valid), 3),
}])

summary_path = OUTDIR / "hygiene_summary.csv"
if summary_path.exists():
    # append without header
    summary.to_csv(summary_path, mode="a", index=False, header=False)
else:
    summary.to_csv(summary_path, index=False)

print("Wrote summary →", summary_path)
summary


Wrote summary → /content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/GenAI/Filtering/2-medchem_hygiene/mao-b_hygiene-20251225-094837/hygiene_summary.csv


,timestamp,target,input_file,annotated_file,pass_smi_file,n_input,n_valid,n_pass,n_alert_flagged,pass_rate_vs_input,pass_rate_vs_valid
0,2025-12-25 09:48:40,Filtering,/content/drive/MyDrive/Colab Notebooks/ESOFT-M...,/content/drive/MyDrive/Colab Notebooks/ESOFT-M...,/content/drive/MyDrive/Colab Notebooks/ESOFT-M...,895,895,327,527,0.365,0.365
